# JJP merge 后 Ntuple 研究 Notebook

这个 Notebook 面向已经完成 merge 的 JJP data 样本，默认输入是 `selected` tree。

## merge 阶段已经预加的 cut

- 只保留 `bestCandIdx >= 0` 的候选。
- 事件内多候选时，按 `sqrt(pt1^2 + pt2^2 + pt3^2)` 选择最佳候选。
- J/psi 质量窗口：`2.9 < m(J/psi) < 3.3`。
- Phi 质量窗口：`0.99 < m(Phi) < 1.07`。
- `pT(J/psi_1) > 6 GeV`，`pT(J/psi_2) > 6 GeV`，`pT(Phi) > 4 GeV`。
- `|y(J/psi)| < 2.5`。
- Kaon 要求：`pT > 2 GeV`，`|eta| < 2.5`。
- Muon 重建运动学要求：`|eta| < 1.2` 时 `pT > 3.5 GeV`；`1.2 < |eta| < 2.4` 时 `pT > 2.5 GeV`。
- Muon ID 在 merge 时默认使用 `soft`；如果你实际 merge 时改过 `--muon-id`，以实际 merge 配置为准。

## merge 阶段没有预加，但在本 Notebook 中可继续追加的条件

- trigger 条件。
- 原始 ntuple 中保留的 muon ID / muon quality / trigger match / vertex / ctau / chi2 / ndof / vtxProb 等条件。
- 更严格的动力学 cut、信号区/旁带 cut、以及自定义物理量 cut。


## 0. 环境初始化

这一段完成以下工作：

- 载入 `CMSSW` 环境下常用的 Python 包与 `ROOT`。
- 开启 `ROOT` 隐式多线程。
- 设置 `mplhep` 的 CMS 标准绘图风格。
- 导入仓库中现有的 `fit_splot.py` 和 `plot_weighted_distributions.py` 里的核心函数，避免重复实现。

如果 Notebook kernel 不是从当前 `CMSSW` 环境启动的，请先在终端里完成 `cmsenv` 再启动 Jupyter。

In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
import tempfile
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from glob import glob

def extend_cmssw_python_paths():
    base = '/cvmfs/cms.cern.ch/el9_amd64_gcc12/external'
    package_globs = [
        'py3-awkward/*/lib/python3.9/site-packages',
        'py3-awkward-cpp/*/lib/python3.9/site-packages',
        'py3-uproot/*/lib/python3.9/site-packages',
        'py3-mplhep/*/lib/python3.9/site-packages',
        'py3-mplhep-data/*/lib/python3.9/site-packages',
    ]
    extra_paths = []
    for pattern in package_globs:
        matches = sorted(glob(os.path.join(base, pattern)))
        if matches:
            extra_paths.append(matches[-1])
    for p in reversed(extra_paths):
        if p not in sys.path:
            sys.path.insert(0, p)
    return extra_paths

EXTRA_PATHS = extend_cmssw_python_paths()

import awkward as ak
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np
import pandas as pd
import ROOT
import uproot

PROJECT_DIR = Path('/eos/home-x/xcheng/CMSSW_15_0_15/src/NtupleAnalyzer')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from ntuple_pipeline_common import CHANNEL_CONFIGS, ensure_dir
from fit_splot import (
    INPUT_TREE as FIT_INPUT_TREE,
    build_jjp_model,
    build_jup_model,
    clone_tree_with_weights,
    make_dataset,
    save_projection_plots,
)
from plot_weighted_distributions import infer_histogram

THREADS = max(1, min(8, os.cpu_count() or 1))
ROOT.EnableImplicitMT(THREADS)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)
ROOT.gROOT.SetBatch(True)

plt.style.use(hep.style.CMS)
plt.rcParams.update({
    'figure.figsize': (8.0, 6.0),
    'axes.grid': True,
    'grid.alpha': 0.25,
    'savefig.dpi': 150,
    'figure.dpi': 110,
})

UPROOT_DECOMP_EXECUTOR = ThreadPoolExecutor(max_workers=THREADS)
UPROOT_INTERP_EXECUTOR = ThreadPoolExecutor(max_workers=THREADS)

NOTEBOOK_CONFIG = {
    'channel': 'JJP',
    'tree_name': 'selected',
    'input_file': '/eos/user/x/xcheng/learn_MC/NtupleAnalyzer_assocPV/merged/jjp_data_selected.root',
    'work_dir': str(PROJECT_DIR / 'notebook_outputs' / 'jjp_data_study'),
    'threads': THREADS,
    'uproot_step_size': '200 MB',
}
ensure_dir(NOTEBOOK_CONFIG['work_dir'])
print(json.dumps(NOTEBOOK_CONFIG, ensure_ascii=False, indent=2))


## 1. 读入 merge 后的 Ntuple（`uproot` 优化读取）

这一段提供几个实用函数：

- 快速查看 tree 和 branch 概览。
- 自动把 branch 分成 `sel_*`、trigger、muon ID、vertex 等几类。
- 使用 `uproot` 多线程读取，并支持只读取感兴趣的 branch。
- 如果样本很大，可以用 `iterate` 分块读取，而不是一次性全读入内存。

In [ ]:
def open_selected_tree(input_file: str | None = None, tree_name: str | None = None):
    input_file = input_file or NOTEBOOK_CONFIG['input_file']
    tree_name = tree_name or NOTEBOOK_CONFIG['tree_name']
    return uproot.open(input_file)[tree_name]


def categorize_branches(branch_names):
    branch_names = sorted(branch_names)
    return {
        'sel_scalar': [b for b in branch_names if b.startswith('sel_')],
        'trigger': [b for b in branch_names if b.startswith(('HLT_', 'L1_'))],
        'muon_id': [b for b in branch_names if 'muIsPat' in b or 'Muon' in b or 'GlobalMuon' in b or 'TrackerMuon' in b],
        'vertex': [b for b in branch_names if 'Vtx' in b or 'vtx' in b or 'ctau' in b or 'Chi2' in b or 'ndof' in b],
        'other': [b for b in branch_names if not (
            b.startswith('sel_') or b.startswith(('HLT_', 'L1_')) or 'muIsPat' in b or 'Muon' in b or 'Vtx' in b or 'vtx' in b or 'ctau' in b or 'Chi2' in b or 'ndof' in b
        )],
    }


def show_tree_summary(input_file: str | None = None, tree_name: str | None = None, max_branches: int = 15):
    tree = open_selected_tree(input_file=input_file, tree_name=tree_name)
    branch_groups = categorize_branches(tree.keys())
    summary = pd.DataFrame({
        'group': list(branch_groups.keys()),
        'n_branches': [len(v) for v in branch_groups.values()],
        'examples': [', '.join(v[:max_branches]) for v in branch_groups.values()],
    })
    display(summary)
    print(f'entries = {tree.num_entries:,}')
    return tree, branch_groups


def build_analysis_branch_list(tree=None, include_all: bool = False, extra_branches: list[str] | None = None):
    tree = tree or open_selected_tree()
    groups = categorize_branches(tree.keys())
    if include_all:
        branch_list = list(tree.keys())
    else:
        branch_list = sorted(set(
            groups['sel_scalar']
            + groups['trigger']
            + groups['muon_id']
            + groups['vertex']
            + ['bestCandIdx', 'muPx', 'muPy', 'muPz']
        ))
    if extra_branches:
        branch_list = sorted(set(branch_list + list(extra_branches)))
    return branch_list


def load_ntuple_arrays(
    input_file: str | None = None,
    tree_name: str | None = None,
    branches: list[str] | None = None,
    cut: str | None = None,
    library: str = 'ak',
):
    tree = open_selected_tree(input_file=input_file, tree_name=tree_name)
    return tree.arrays(
        expressions=branches,
        cut=cut,
        library=library,
        decompression_executor=UPROOT_DECOMP_EXECUTOR,
        interpretation_executor=UPROOT_INTERP_EXECUTOR,
    )


def iterate_ntuple_arrays(
    input_file: str | None = None,
    tree_name: str | None = None,
    branches: list[str] | None = None,
    cut: str | None = None,
    library: str = 'ak',
    step_size: str | int | None = None,
):
    tree = open_selected_tree(input_file=input_file, tree_name=tree_name)
    step_size = step_size or NOTEBOOK_CONFIG['uproot_step_size']
    yield from tree.iterate(
        expressions=branches,
        cut=cut,
        library=library,
        step_size=step_size,
        decompression_executor=UPROOT_DECOMP_EXECUTOR,
        interpretation_executor=UPROOT_INTERP_EXECUTOR,
    )


TREE, BRANCH_GROUPS = show_tree_summary()
ANALYSIS_BRANCHES = build_analysis_branch_list(TREE)
print(f'analysis branches = {len(ANALYSIS_BRANCHES)}')


## 2. 对动力学量继续加 cut（参数可调）

这一段把后处理 cut 抽象成统一配置，可以同时支持：

- 标量物理量范围 cut，例如 `sel_Jpsi_1_pt`、`sel_Phi_VtxProb`。
- 任意 trigger 分支要求至少一个通过。
- 从原始 jagged muon ID 分支中，按 `sel_*_mu_*_Idx` 回取所选 muon 的 ID flag，再做 cut。
- 输出 cutflow，方便你比较每一步损失的事件数。

这里默认给出一套和 merge 相近、但可继续收紧的示例配置。你可以直接改字典里的数值或新增分支。

In [ ]:
def record_length(arrays) -> int:
    return len(arrays[arrays.fields[0]]) if arrays.fields else 0


def take_jagged_by_selected_index(jagged_values, selected_indices, default=0):
    selected_indices = ak.values_astype(selected_indices, np.int64)
    local_index = ak.local_index(jagged_values)
    picked = ak.firsts(jagged_values[local_index == selected_indices])
    return ak.fill_none(picked, default)


def as_numpy(values):
    return ak.to_numpy(ak.fill_none(values, np.nan))


def apply_scalar_cut(mask, arrays, branch, spec):
    values = as_numpy(arrays[branch])
    branch_mask = np.ones(len(values), dtype=bool)
    if spec.get('min') is not None:
        branch_mask &= values >= spec['min']
    if spec.get('max') is not None:
        branch_mask &= values <= spec['max']
    if spec.get('abs_min') is not None:
        branch_mask &= np.abs(values) >= spec['abs_min']
    if spec.get('abs_max') is not None:
        branch_mask &= np.abs(values) <= spec['abs_max']
    if spec.get('eq') is not None:
        branch_mask &= values == spec['eq']
    return mask & branch_mask


def apply_trigger_cut(mask, arrays, trigger_branches, min_passed=1):
    if not trigger_branches:
        return mask
    trigger_sum = np.zeros(record_length(arrays), dtype=np.int32)
    for branch in trigger_branches:
        trigger_sum += as_numpy(arrays[branch]).astype(np.int32)
    return mask & (trigger_sum >= min_passed)


def apply_indexed_flag_cut(mask, arrays, spec):
    values = take_jagged_by_selected_index(arrays[spec['flag_branch']], arrays[spec['index_branch']], default=0)
    values = ak.to_numpy(values).astype(np.int32)
    return mask & (values == spec.get('value', 1))


def apply_cut_config(arrays, cut_config):
    mask = np.ones(record_length(arrays), dtype=bool)
    cutflow_rows = [('initial', int(mask.sum()))]

    for branch, spec in cut_config.get('scalar_cuts', OrderedDict()).items():
        mask = apply_scalar_cut(mask, arrays, branch, spec)
        cutflow_rows.append((branch, int(mask.sum())))

    if cut_config.get('require_any_trigger', False):
        mask = apply_trigger_cut(
            mask,
            arrays,
            cut_config.get('trigger_branches', []),
            cut_config.get('min_trigger_passed', 1),
        )
        cutflow_rows.append(('trigger', int(mask.sum())))

    for spec in cut_config.get('indexed_flag_cuts', []):
        mask = apply_indexed_flag_cut(mask, arrays, spec)
        cutflow_rows.append((spec.get('name', spec['flag_branch']), int(mask.sum())))

    filtered = arrays[mask]
    cutflow = pd.DataFrame(cutflow_rows, columns=['cut', 'events'])
    cutflow['eff_rel'] = cutflow['events'] / cutflow['events'].shift(1).fillna(cutflow['events'].iloc[0])
    cutflow['eff_abs'] = cutflow['events'] / cutflow['events'].iloc[0]
    return filtered, cutflow, mask


SUGGESTED_TRIGGER_BRANCHES = [
    name for name in BRANCH_GROUPS['trigger']
    if ('Jpsi' in name) or ('Quarkonium' in name) or ('Dimuon' in name)
]

CUT_CONFIG = {
    'scalar_cuts': OrderedDict({
        'sel_Jpsi_1_pt': {'min': 6.0},
        'sel_Jpsi_2_pt': {'min': 6.0},
        'sel_Phi_pt': {'min': 4.0},
        'sel_Jpsi_1_VtxProb': {'min': 0.05},
        'sel_Jpsi_2_VtxProb': {'min': 0.05},
        'sel_Phi_VtxProb': {'min': 0.05},
        'sel_Jpsi_1_mass': {'min': 2.9, 'max': 3.3},
        'sel_Jpsi_2_mass': {'min': 2.9, 'max': 3.3},
        'sel_Phi_mass': {'min': 0.99, 'max': 1.07},
    }),
    'require_any_trigger': False,
    'trigger_branches': SUGGESTED_TRIGGER_BRANCHES[:4],
    'min_trigger_passed': 1,
    'indexed_flag_cuts': [
        {'name': 'jpsi1_mu1_soft', 'flag_branch': 'muIsPatSoftMuon', 'index_branch': 'sel_Jpsi_1_mu_1_Idx', 'value': 1},
        {'name': 'jpsi1_mu2_soft', 'flag_branch': 'muIsPatSoftMuon', 'index_branch': 'sel_Jpsi_1_mu_2_Idx', 'value': 1},
        {'name': 'jpsi2_mu1_soft', 'flag_branch': 'muIsPatSoftMuon', 'index_branch': 'sel_Jpsi_2_mu_1_Idx', 'value': 1},
        {'name': 'jpsi2_mu2_soft', 'flag_branch': 'muIsPatSoftMuon', 'index_branch': 'sel_Jpsi_2_mu_2_Idx', 'value': 1},
    ],
}

print('suggested triggers =', SUGGESTED_TRIGGER_BRANCHES[:10])


## 2.1 执行 cut，并导出过滤后的 ROOT 文件

这一段会：

- 用上面的 branch 列表和 cut 配置读取数据。
- 输出 cutflow 表。
- 把 cut 后样本写成新的 `selected` tree，供后面的 RooFit / sPlot 直接使用。

如果你希望保留更多原始 branch，只需要把 `INCLUDE_ALL_BRANCHES_FOR_EXPORT` 设成 `True`。

In [ ]:
def write_filtered_root(arrays, output_file: str | Path, tree_name: str = 'selected'):
    output_file = str(output_file)
    ensure_dir(str(Path(output_file).parent))
    payload = {name: ak.to_packed(arrays[name]) for name in arrays.fields}
    with uproot.recreate(output_file) as fout:
        fout[tree_name] = payload
    return output_file


INCLUDE_ALL_BRANCHES_FOR_EXPORT = False
EXPORT_FILE = Path(NOTEBOOK_CONFIG['work_dir']) / 'jjp_filtered_selected.root'

READ_BRANCHES = build_analysis_branch_list(TREE, include_all=INCLUDE_ALL_BRANCHES_FOR_EXPORT)
RAW_ARRAYS = load_ntuple_arrays(branches=READ_BRANCHES)
FILTERED_ARRAYS, CUTFLOW, EVENT_MASK = apply_cut_config(RAW_ARRAYS, CUT_CONFIG)
display(CUTFLOW)
FILTERED_FILE = write_filtered_root(FILTERED_ARRAYS, EXPORT_FILE, tree_name=NOTEBOOK_CONFIG['tree_name'])
print(FILTERED_FILE)
print(f'filtered events = {record_length(FILTERED_ARRAYS):,}')


## 3. 三维 sPlot 拟合并提取 sWeight

这里默认直接复用仓库里 `fit_splot.py` 的 3D 模型：

- JJP: `m(J/psi_1) × m(J/psi_2) × m(Phi)`。
- 分量为 `sss / ssb / sbs / bss / sbb / bsb / bbs / bbb` 共 8 个扩展项。
- `signal_sw` 默认对应 `yield_sss_sw`。

为了方便交互研究，这里把模型参数、观察量范围、是否固定参数都做成可调配置。
如果你想完全换模型，可以把 `model_builder` 换成你自己的函数，只要返回格式与 `build_jjp_model` 一致即可。

In [ ]:
def build_model_with_map(channel: str, n_events: int, model_builder=None):
    channel = channel.upper()
    if model_builder is None:
        model_builder = build_jjp_model if channel == 'JJP' else build_jup_model
    model, observables, yields, signal_yield_name, keepalive = model_builder(n_events)
    param_map = {}
    for obj in keepalive:
        if hasattr(obj, 'GetName'):
            param_map[obj.GetName()] = obj
    for name, obs in observables.items():
        param_map[name] = obs
    for name, yld in yields.items():
        param_map[name] = yld
    return model, observables, yields, signal_yield_name, keepalive, param_map


def apply_fit_overrides(observables, param_map, fit_config):
    for name, bounds in fit_config.get('observable_ranges', {}).items():
        if name in observables:
            observables[name].setRange(bounds[0], bounds[1])

    for name, value in fit_config.get('set_values', {}).items():
        if name in param_map and hasattr(param_map[name], 'setVal'):
            param_map[name].setVal(value)

    for name, bounds in fit_config.get('set_ranges', {}).items():
        if name in param_map:
            if hasattr(param_map[name], 'setMin'):
                param_map[name].setMin(bounds[0])
            if hasattr(param_map[name], 'setMax'):
                param_map[name].setMax(bounds[1])

    for name in fit_config.get('freeze', []):
        if name in param_map and hasattr(param_map[name], 'setConstant'):
            param_map[name].setConstant(True)


def run_splot_fit(
    input_file: str,
    channel: str = 'JJP',
    output_file: str | None = None,
    plot_dir: str | None = None,
    fit_config: dict | None = None,
    model_builder=None,
):
    channel = channel.upper()
    fit_config = fit_config or {}
    output_file = output_file or str(Path(NOTEBOOK_CONFIG['work_dir']) / 'jjp_filtered_weighted.root')
    plot_dir = plot_dir or str(Path(NOTEBOOK_CONFIG['work_dir']) / 'fit_plots')
    ensure_dir(plot_dir)

    tree = uproot.open(input_file)[FIT_INPUT_TREE]
    n_entries = tree.num_entries
    model, observables, yields, signal_yield_name, keepalive, param_map = build_model_with_map(
        channel=channel,
        n_events=n_entries,
        model_builder=model_builder,
    )
    apply_fit_overrides(observables, param_map, fit_config)

    fin = ROOT.TFile.Open(input_file)
    root_tree = fin.Get(FIT_INPUT_TREE)
    data = make_dataset(root_tree, observables)
    keepalive.append(data)

    fit_result = model.fitTo(
        data,
        ROOT.RooFit.Extended(True),
        ROOT.RooFit.Save(True),
        ROOT.RooFit.NumCPU(max(1, int(fit_config.get('jobs', THREADS)))),
        ROOT.RooFit.Strategy(int(fit_config.get('strategy', 1))),
        ROOT.RooFit.PrintLevel(int(fit_config.get('print_level', -1))),
    )
    keepalive.append(fit_result)

    save_projection_plots(channel, plot_dir, data, model, observables, signal_yield_name)
    sdata = ROOT.RooStats.SPlot('sData', 'sData', data, model, ROOT.RooArgList(*yields.values()))
    keepalive.append(sdata)

    weight_map = OrderedDict()
    for yield_name in yields:
        weight_map[f'{yield_name}_sw'] = [data.get(i).getRealValue(f'{yield_name}_sw') for i in range(data.numEntries())]
    weight_map['signal_sw'] = list(weight_map[f'{signal_yield_name}_sw'])

    clone_tree_with_weights(input_file, output_file, weight_map)

    fit_result_file = output_file.replace('.root', '_fit_result.root')
    fout = ROOT.TFile(fit_result_file, 'RECREATE')
    fit_result.Write('fit_result')
    fout.Close()
    fin.Close()

    yield_summary = pd.DataFrame({
        'yield_name': list(yields.keys()),
        'value': [y.getVal() for y in yields.values()],
        'error': [y.getError() if hasattr(y, 'getError') else np.nan for y in yields.values()],
    })

    return {
        'output_file': output_file,
        'fit_result_file': fit_result_file,
        'plot_dir': plot_dir,
        'signal_yield_name': signal_yield_name,
        'yield_summary': yield_summary,
        'fit_result': fit_result,
        'keepalive': keepalive,
        'observables': observables,
        'yields': yields,
    }

# FILTERED_FILE = '/eos/user/x/xcheng/learn_MC/NtupleAnalyzer_assocPV/merged/jjp_data_selected.root'

FIT_CONFIG = {
    'jobs': THREADS,
    'strategy': 1,
    'print_level': -1,
    'observable_ranges': {
        'sel_Jpsi_1_mass': (2.9, 3.3),
        'sel_Jpsi_2_mass': (2.9, 3.3),
        'sel_Phi_mass': (0.99, 1.07),
    },
    'set_values': {
        'jpsi_mean': 3.096,
        'phi_mean': 1.019,
    },
    'set_ranges': {
        'jpsi_sigma_cb': (0.003, 0.08),
        'jpsi_sigma_gauss': (0.003, 0.12),
        'phi_sigma': (0.0002, 0.02),
    },
    'freeze': [],
}

SPLOT_RESULT = run_splot_fit(
    input_file=FILTERED_FILE,
    channel=NOTEBOOK_CONFIG['channel'],
    output_file=str(Path(NOTEBOOK_CONFIG['work_dir']) / 'jjp_filtered_weighted.root'),
    plot_dir=str(Path(NOTEBOOK_CONFIG['work_dir']) / 'fit_plots'),
    fit_config=FIT_CONFIG,
)
display(SPLOT_RESULT['yield_summary'])
print('weighted file =', SPLOT_RESULT['output_file'])


## 4. 用 sWeight 作图（`mplhep` CMS style）

这里使用 `mplhep` 统一画图风格，和前面的 RooFit 投影图区分开。

功能包括：

- 任意 `sel_*` 标量分支的加权 1D 直方图。
- `Δy` / `Δφ` 的比较图。
- 二维相关图。
- 自动遍历 `weighted.root` 里的标量 `sel_*` 分支并批量输出。

如果需要和原脚本一致的 binning，会复用 `plot_weighted_distributions.py` 中的 `infer_histogram()`。

In [ ]:
def discover_scalar_sel_branches(weighted_file: str, tree_name: str = 'selected'):
    tree = uproot.open(weighted_file)[tree_name]
    type_map = tree.typenames()
    branches = []
    for name, typename in type_map.items():
        if not name.startswith('sel_'):
            continue
        if '[' in typename or 'std::vector' in typename:
            continue
        if name.endswith('_Idx'):
            continue
        branches.append(name)
    return sorted(set(branches))


def load_weighted_arrays(weighted_file: str, branches: list[str]):
    return load_ntuple_arrays(input_file=weighted_file, branches=branches)


def cms_label(ax, data_label='Private Work'):
    hep.cms.label(ax=ax, data=True, label=data_label)


def make_weighted_hist(values, weights, bins, xmin, xmax, density=False):
    hist, edges = np.histogram(values, bins=bins, range=(xmin, xmax), weights=weights, density=density)
    return hist, edges


def plot_weighted_1d(arrays, branch, weight_branch='signal_sw', output_dir='.', density=False):
    bins, xmin, xmax = infer_histogram(branch)
    values = as_numpy(arrays[branch])
    weights = as_numpy(arrays[weight_branch])
    valid = np.isfinite(values) & np.isfinite(weights)
    hist, edges = make_weighted_hist(values[valid], weights[valid], bins, xmin, xmax, density=density)

    fig, ax = plt.subplots()
    hep.histplot(hist, edges, ax=ax, histtype='step', linewidth=2, color='tab:blue')
    ax.set_xlabel(branch)
    ax.set_ylabel('Weighted events' if not density else 'A.U.')
    cms_label(ax)
    fig.tight_layout()
    ensure_dir(output_dir)
    fig.savefig(Path(output_dir) / f'{branch}.png')
    fig.savefig(Path(output_dir) / f'{branch}.pdf')
    plt.close(fig)


def plot_overlay(arrays, branches, weight_branch='signal_sw', output_name='overlay', output_dir='.'):
    fig, ax = plt.subplots()
    colors = ['tab:red', 'tab:blue', 'tab:green']
    for branch, color in zip(branches, colors):
        bins, xmin, xmax = infer_histogram(branch)
        values = as_numpy(arrays[branch])
        weights = as_numpy(arrays[weight_branch])
        valid = np.isfinite(values) & np.isfinite(weights)
        hist, edges = make_weighted_hist(values[valid], weights[valid], bins, xmin, xmax)
        hep.histplot(hist, edges, ax=ax, histtype='step', linewidth=2, label=branch.replace('sel_', ''), color=color)
    ax.legend(frameon=False)
    ax.set_ylabel('Weighted events')
    cms_label(ax)
    fig.tight_layout()
    ensure_dir(output_dir)
    fig.savefig(Path(output_dir) / f'{output_name}.png')
    fig.savefig(Path(output_dir) / f'{output_name}.pdf')
    plt.close(fig)


def plot_weighted_2d(arrays, x_branch, y_branch, weight_branch='signal_sw', output_name='corr2d', output_dir='.'):
    x = as_numpy(arrays[x_branch])
    y = as_numpy(arrays[y_branch])
    w = as_numpy(arrays[weight_branch])
    valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(w)

    fig, ax = plt.subplots()
    h = ax.hist2d(x[valid], y[valid], bins=(60, 60), weights=w[valid], cmap='viridis')
    ax.set_xlabel(x_branch)
    ax.set_ylabel(y_branch)
    cms_label(ax)
    cbar = fig.colorbar(h[3], ax=ax)
    cbar.set_label('Weighted events')
    fig.tight_layout()
    ensure_dir(output_dir)
    fig.savefig(Path(output_dir) / f'{output_name}.png')
    fig.savefig(Path(output_dir) / f'{output_name}.pdf')
    plt.close(fig)


def plot_standard_suite(weighted_file: str, channel: str = 'JJP', output_dir: str | None = None, weight_branch: str = 'signal_sw'):
    output_dir = output_dir or str(Path(NOTEBOOK_CONFIG['work_dir']) / 'mplhep_plots')
    scalar_sel = discover_scalar_sel_branches(weighted_file)
    needed = sorted(set(scalar_sel + [weight_branch]))
    arrays = load_weighted_arrays(weighted_file, branches=needed)

    for branch in scalar_sel:
        plot_weighted_1d(arrays, branch, weight_branch=weight_branch, output_dir=output_dir)

    cfg = CHANNEL_CONFIGS[channel.upper()]
    dy_branches = [f'sel_abs_dy_{name}' for name, _, _ in cfg.pair_specs]
    dphi_branches = [f'sel_abs_dphi_{name}' for name, _, _ in cfg.pair_specs]
    plot_overlay(arrays, dy_branches, weight_branch=weight_branch, output_name='delta_y_comparison', output_dir=output_dir)
    plot_overlay(arrays, dphi_branches, weight_branch=weight_branch, output_name='delta_phi_comparison', output_dir=output_dir)

    for pair_name, _, _ in cfg.pair_specs:
        plot_weighted_2d(
            arrays,
            x_branch=f'sel_abs_dy_{pair_name}',
            y_branch=f'sel_abs_dphi_{pair_name}',
            weight_branch=weight_branch,
            output_name=f'correlation_2d_{pair_name}',
            output_dir=output_dir,
        )
    return output_dir


PLOT_DIR = plot_standard_suite(
    weighted_file=SPLOT_RESULT['output_file'],
    channel=NOTEBOOK_CONFIG['channel'],
    output_dir=str(Path(NOTEBOOK_CONFIG['work_dir']) / 'mplhep_plots'),
    weight_branch='signal_sw',
)
print(PLOT_DIR)


## 5. 建议的交互式使用顺序

你后续通常按下面顺序改就够了：

1. 在第 2 节修改 `CUT_CONFIG`，决定 trigger、muon ID、vertex 和动力学 cut。
2. 重新运行第 2.1 节，得到新的 `FILTERED_FILE` 和 cutflow。
3. 在第 3 节修改 `FIT_CONFIG`，决定拟合参数初值、范围、冻结策略，必要时替换 `model_builder`。
4. 重新运行第 3 节拿到新的 `signal_sw`。
5. 重新运行第 4 节，输出新的加权分布图。

如果你想先只研究 cut 而不做 sPlot，那么可以只运行到第 2.1 节。